In [1]:
"""
Unified Mortgage Default Modeling Pipeline
Combines the three submitted notebooks into one reproducible script.

Experiments included:
1. Data cleaning, sampling, target construction, and temporal feature engineering
2. Loan-level split and Out-of-Time (OOT) split
3. Raw/tabular Random Forest baseline
4. 12-month sequential Random Forest baseline
5. Threshold analysis
6. SMOTE comparison
7. Logistic Regression feature ablation
8. Logistic Regression imbalance comparison
9. Logistic Regression regularization sweep
10. Full model shootout: Logistic Regression, Random Forest, XGBoost, LightGBM, GRU, LSTM
11. Feature ablation: XGBoost + LSTM
12. Calibration curves
13. LSTM temporal sensitivity: 3, 6, 12, 24 months
14. ROC/PR curves, confusion matrices, feature importance, and CSV exports

IMPORTANT:
- The target is "next-month serious delinquency": use the previous N months as
  features and predict serious delinquency at the following month.
- A serious delinquency is 90+ days delinquent. "R" is treated as serious
  delinquency; "XX" is treated as missing.
- All scalers/imputers/SMOTE are fit on training data only.
- Loan-level splits prevent the same loan from appearing in train and test.
- OOT splits use chronological target months.
"""

import os
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix, roc_curve,
    precision_recall_curve, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve

# Optional packages used by the original notebooks.
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    SMOTE = None

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import (
        LSTM, GRU, Dense, Dropout, BatchNormalization, LeakyReLU
    )
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
except ImportError:
    tf = None

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')




# ============================================================
# 0. CONFIGURATION
# ============================================================

DATA_PATH = "/content/merged_mortgage_2013.csv"
#RESULTS_DIR = Path("results_unified")

#RESULTS_DIR = Path("results_unified_full")
RESULTS_DIR = Path("/content/drive/MyDrive/results_unified_full")


RANDOM_STATE = 42
#N_LOANS = 6000             # Increase if memory/time allows.

N_LOANS = 1000000          # Set to 1 million to force it to use all available loans

SEQUENCE_LENGTH = 12
TEMPORAL_SEQUENCE_LENGTHS = [3, 6, 12, 24]

RUN_RAW_RF = True
RUN_SEQUENCE_RF = True
RUN_LOGISTIC = True
RUN_FULL_SHOOTOUT = True
RUN_FEATURE_ABLATION = True
RUN_TEMPORAL_SENSITIVITY = True
RUN_PLOTS = True
RUN_NOTEBOOK_VISUALS = True
RUN_STRICT_THRESHOLD_REPORT = True
RUN_DATA_QUALITY_REPORT = True

RF_TREES = 200
SEQUENCE_RF_TREES = 300
XGB_TREES = 100
LGBM_TREES = 100

DL_EPOCHS = 15
ABLATION_DL_EPOCHS = 10
DL_BATCH_SIZE = 512

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

if tf is not None:
    tf.random.set_seed(RANDOM_STATE)


# ============================================================
# 1. DATA CLEANING AND TARGET
# ============================================================

DROP_COLUMNS = [
    "defect_settlement_date",
    "legal_costs",
    "mi_recoveries",
    "miscellaneous_expenses",
    "taxes_and_insurance",
    "maintenance_and_preservation_costs",
    "expenses",
    "non_mi_recoveries",
    "delinquent_accrued_interest",
    "net_sales_proceeds",
    "actual_loss_calculation",
    "modification_cost",
    "current_month_modification_cost",
    "harp_indicator",
    "pre_harp_loan_sequence_number",
    "super_conforming_flag",
]

RAW_FEATURES = [
    "current_actual_upb",
    "current_interest_rate",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "current_deferred_upb",
    "interest_bearing_upb",
    "credit_score",
    "mortgage_insurance_percentage",
    "number_of_units",
    "original_combined_loan_to_value",
    "original_debt_to_income_ratio",
    "original_unpaid_principal_balance",
    "original_loan_to_value",
    "original_interest_rate",
    "original_loan_term",
    "number_of_borrowers",
    "estimated_loan_to_value",
]

BASE_TEMPORAL_FEATURES = [
    "normalized_upb",
    "current_interest_rate",
    "loan_age",
    "credit_score",
    "estimated_loan_to_value",
]

FEATURE_SETS = {
    "A_Base": BASE_TEMPORAL_FEATURES,
    "B_Plus_UPB": BASE_TEMPORAL_FEATURES + ["upb_pct_change_1m"],
    "C_Plus_Delinq": BASE_TEMPORAL_FEATURES + [
        "prev_month_delinquency",
        "delinq_count_6m",
    ],
    "D_Full": BASE_TEMPORAL_FEATURES + [
        "upb_pct_change_1m",
        "prev_month_delinquency",
        "delinq_count_6m",
    ],
}


def change_delinquency(status):
    """
    1 = 90+ days delinquent or R/REO-type serious status
    0 = current / fewer than 90 days delinquent
    NaN = unknown / XX
    """
    if pd.isna(status):
        return np.nan

    status = str(status).strip().upper()

    if status == "XX":
        return np.nan

    # The submitted Random Forest notebook treated "R" as serious.
    if status == "R":
        return 1

    try:
        return 1 if int(float(status)) >= 3 else 0
    except (ValueError, TypeError):
        return np.nan


def load_and_prepare_data(path, n_loans=N_LOANS):
    print("=" * 80)
    print("1. LOADING AND PREPARING DATA")
    print("=" * 80)

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path.resolve()}. "
            "Update DATA_PATH at the top of the script."
        )

    print(f"Reading: {path.resolve()}")
    merged = pd.read_csv(path)
    print(f"Original shape: {merged.shape}")

    # Drop columns only when present.
    drop_now = [c for c in DROP_COLUMNS if c in merged.columns]
    if drop_now:
        merged = merged.drop(columns=drop_now)
        print(f"Dropped {len(drop_now)} columns.")

    required = {
        "loan_sequence_number",
        "monthly_reporting_period",
        "current_loan_delinquency_status",
    }
    missing = sorted(required - set(merged.columns))
    if missing:
        raise ValueError(f"Required columns are missing: {missing}")

    # Sample whole loans, never individual rows.
    unique_loans = merged["loan_sequence_number"].dropna().unique()
    sample_size = min(n_loans, len(unique_loans))

    rng = np.random.default_rng(RANDOM_STATE)
    selected_loans = rng.choice(unique_loans, size=sample_size, replace=False)

    df = merged[
        merged["loan_sequence_number"].isin(selected_loans)
    ].copy()

    df = df.sort_values(
        ["loan_sequence_number", "monthly_reporting_period"]
    ).reset_index(drop=True)

    print(f"Working sample: {df.shape[0]:,} rows")
    print(f"Unique loans: {df['loan_sequence_number'].nunique():,}")

    # Binary current-month status.
    df["changeDelinquency"] = (
        df["current_loan_delinquency_status"].apply(change_delinquency)
    )

    # Target = next month's serious delinquency.
    df["next_month_serious_delinquency"] = (
        df.groupby("loan_sequence_number")["changeDelinquency"].shift(-1)
    )

    # Remove rows without a known next-month target.
    df = df.dropna(
        subset=["next_month_serious_delinquency"]
    ).copy()

    df["next_month_serious_delinquency"] = (
        df["next_month_serious_delinquency"].astype(int)
    )

    # This target is the same conceptual target used by the sequence work:
    # historical months -> following month's serious delinquency.
    print("\nTarget distribution:")
    print(
        df["next_month_serious_delinquency"]
        .value_counts()
        .sort_index()
    )
    print(
        df["next_month_serious_delinquency"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(4)
        .rename("percent")
    )

    return df


# ============================================================
# 2. TEMPORAL FEATURE ENGINEERING
# ============================================================

def engineer_temporal_features(df):
    print("\n" + "=" * 80)
    print("2. TEMPORAL FEATURE ENGINEERING")
    print("=" * 80)

    df = df.copy()
    df = df.sort_values(
        ["loan_sequence_number", "monthly_reporting_period"]
    ).reset_index(drop=True)

    # Previous-month delinquency: available before the prediction month.
    df["prev_month_delinquency"] = (
        df.groupby("loan_sequence_number")["changeDelinquency"]
        .shift(1)
        .fillna(0)
    )

    # Rolling delinquency history uses previous-month status.
    df["delinq_count_6m"] = (
        df.groupby("loan_sequence_number")["prev_month_delinquency"]
        .rolling(window=6, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .fillna(0)
    )

    # Normalize UPB.
    if "original_loan_amount" in df.columns:
        denominator = pd.to_numeric(
            df["original_loan_amount"], errors="coerce"
        )
        df["normalized_upb"] = (
            pd.to_numeric(df["current_actual_upb"], errors="coerce")
            / denominator.replace(0, np.nan)
        )
    else:
        first_upb = (
            df.groupby("loan_sequence_number")["current_actual_upb"]
            .transform("first")
        )
        df["normalized_upb"] = (
            pd.to_numeric(df["current_actual_upb"], errors="coerce")
            / pd.to_numeric(first_upb, errors="coerce").replace(0, np.nan)
        )

    # Month-over-month UPB momentum.
    df["upb_pct_change_1m"] = (
        df.groupby("loan_sequence_number")["current_actual_upb"]
        .pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # Convert engineered model columns to numeric.
    all_model_features = sorted(
        set(
            RAW_FEATURES
            + BASE_TEMPORAL_FEATURES
            + [
                "upb_pct_change_1m",
                "prev_month_delinquency",
                "delinq_count_6m",
            ]
        )
    )

    for col in all_model_features:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Date used for OOT splitting.
    df["date_dt"] = pd.to_datetime(
        df["monthly_reporting_period"].astype(str),
        format="%Y%m",
        errors="coerce"
    )

    # Check monthly continuity for diagnostics.
    df["month_diff_days"] = (
        df.groupby("loan_sequence_number")["date_dt"]
        .diff()
        .dt.days
    )

    gaps = df[df["month_diff_days"] > 32]
    print(
        f"Irregular monthly transitions (>32 days): {len(gaps):,}"
    )

    print(
        f"Final positive rate: "
        f"{df['next_month_serious_delinquency'].mean():.4%}"
    )

    save_data_quality_report(df)
    return df


# ============================================================
# 3. SEQUENCE CREATION
# ============================================================

def create_sequences(
    data,
    feature_columns,
    target_column="next_month_serious_delinquency",
    sequence_length=12,
):
    """
    Creates:
      X shape = (samples, sequence_length, features)
      y = target at the month immediately after the sequence.

    The final target month is NOT included inside the feature window.
    """

    sequences = []
    targets = []
    loan_ids = []
    target_months = []

    required = (
        ["loan_sequence_number", "monthly_reporting_period", target_column]
        + feature_columns
    )

    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(
            f"Missing columns needed for sequences: {missing}"
        )

    data = data.sort_values(
        ["loan_sequence_number", "monthly_reporting_period"]
    ).copy()

    for loan_id, group in data.groupby(
        "loan_sequence_number",
        sort=False
    ):
        group = group.reset_index(drop=True)

        if len(group) < sequence_length + 1:
            continue

        feature_values = (
            group[feature_columns]
            .apply(pd.to_numeric, errors="coerce")
            .to_numpy(dtype=np.float32)
        )

        target_values = pd.to_numeric(
            group[target_column],
            errors="coerce"
        ).to_numpy()

        months = group["monthly_reporting_period"].to_numpy()

        for start in range(len(group) - sequence_length):
            target_index = start + sequence_length
            target = target_values[target_index]

            if pd.isna(target):
                continue

            window = feature_values[start:target_index]

            sequences.append(window)
            targets.append(int(target))
            loan_ids.append(loan_id)
            target_months.append(months[target_index])

    if not sequences:
        return (
            np.empty((0, sequence_length, len(feature_columns)), dtype=np.float32),
            np.empty(0, dtype=np.int8),
            np.empty(0, dtype=object),
            np.empty(0, dtype=object),
        )

    return (
        np.asarray(sequences, dtype=np.float32),
        np.asarray(targets, dtype=np.int8),
        np.asarray(loan_ids),
        np.asarray(target_months),
    )


def flatten_sequences(X):
    return X.reshape(X.shape[0], -1)


# ============================================================
# 4. SPLIT FUNCTIONS
# ============================================================

def loan_level_split_sequences(df, feature_list, sequence_length=12):
    """
    64/16/20 loan-level split, matching the original sequential notebook.
    No loan appears in more than one split.
    """
    loan_ids = df["loan_sequence_number"].dropna().unique()

    train_ids, test_ids = train_test_split(
        loan_ids,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    train_ids, val_ids = train_test_split(
        train_ids,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    train_df = df[df["loan_sequence_number"].isin(train_ids)]
    val_df = df[df["loan_sequence_number"].isin(val_ids)]
    test_df = df[df["loan_sequence_number"].isin(test_ids)]

    X_train, y_train, ids_train, months_train = create_sequences(
        train_df, feature_list, sequence_length=sequence_length
    )
    X_val, y_val, ids_val, months_val = create_sequences(
        val_df, feature_list, sequence_length=sequence_length
    )
    X_test, y_test, ids_test, months_test = create_sequences(
        test_df, feature_list, sequence_length=sequence_length
    )

    return {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "train_ids": ids_train,
        "val_ids": ids_val,
        "test_ids": ids_test,
        "train_months": months_train,
        "val_months": months_val,
        "test_months": months_test,
    }


def oot_split_sequences(df, feature_list, sequence_length=12):
    """
    Strict chronological split:
      first 70% of target months = train
      next 15% = validation
      final 15% = test

    This is an out-of-time test of future months.
    """
    X_all, y_all, ids_all, months_all = create_sequences(
        df,
        feature_list,
        sequence_length=sequence_length,
    )

    if len(X_all) == 0:
        raise ValueError("No sequences were generated.")

    month_numeric = pd.to_numeric(
        pd.Series(months_all).astype(str),
        errors="coerce"
    ).to_numpy()

    unique_months = np.sort(
        pd.Series(month_numeric).dropna().unique()
    )

    if len(unique_months) < 5:
        raise ValueError(
            "Not enough unique reporting months for an OOT split."
        )

    train_cutoff = unique_months[
        int(len(unique_months) * 0.70)
    ]
    val_cutoff = unique_months[
        int(len(unique_months) * 0.85)
    ]

    train_mask = month_numeric < train_cutoff
    val_mask = (
        (month_numeric >= train_cutoff)
        & (month_numeric < val_cutoff)
    )
    test_mask = month_numeric >= val_cutoff

    return {
        "X_train": X_all[train_mask],
        "X_val": X_all[val_mask],
        "X_test": X_all[test_mask],
        "y_train": y_all[train_mask],
        "y_val": y_all[val_mask],
        "y_test": y_all[test_mask],
        "train_ids": ids_all[train_mask],
        "val_ids": ids_all[val_mask],
        "test_ids": ids_all[test_mask],
        "train_months": months_all[train_mask],
        "val_months": months_all[val_mask],
        "test_months": months_all[test_mask],
        "cutoffs": (train_cutoff, val_cutoff),
    }


def prepare_data(df, feature_list, split_type, sequence_length=12):
    if split_type == "LOAN_LEVEL":
        data = loan_level_split_sequences(
            df, feature_list, sequence_length
        )
    elif split_type == "OOT":
        data = oot_split_sequences(
            df, feature_list, sequence_length
        )
    else:
        raise ValueError("split_type must be LOAN_LEVEL or OOT")

    # Imputation/scaling are fit only on training data.
    imputer = SimpleImputer(strategy="median")

    n_features = len(feature_list)
    X_train_flat = data["X_train"].reshape(-1, n_features)
    X_val_flat = data["X_val"].reshape(-1, n_features)
    X_test_flat = data["X_test"].reshape(-1, n_features)

    X_train_flat = imputer.fit_transform(X_train_flat)
    X_val_flat = imputer.transform(X_val_flat)
    X_test_flat = imputer.transform(X_test_flat)

    scaler = MinMaxScaler()
    X_train_flat = scaler.fit_transform(X_train_flat)
    X_val_flat = scaler.transform(X_val_flat)
    X_test_flat = scaler.transform(X_test_flat)

    data["X_train_3d"] = X_train_flat.reshape(
        -1, sequence_length, n_features
    )
    data["X_val_3d"] = X_val_flat.reshape(
        -1, sequence_length, n_features
    )
    data["X_test_3d"] = X_test_flat.reshape(
        -1, sequence_length, n_features
    )

    data["X_train_2d"] = flatten_sequences(data["X_train_3d"])
    data["X_val_2d"] = flatten_sequences(data["X_val_3d"])
    data["X_test_2d"] = flatten_sequences(data["X_test_3d"])

    data["imputer"] = imputer
    data["scaler"] = scaler

    return data


# ============================================================
# 5. METRICS AND THRESHOLDS
# ============================================================

def threshold_metrics(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)

    return {
        "Threshold": threshold,
        "Precision": precision_score(
            y_true, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, pred, zero_division=0
        ),
        "Accuracy": accuracy_score(y_true, pred),
    }


def evaluate_model(
    y_true,
    probabilities,
    model_name,
    split_name,
    feature_set="D_Full",
):
    probabilities = np.clip(
        np.asarray(probabilities).ravel(), 0, 1
    )

    roc_auc = roc_auc_score(y_true, probabilities)
    pr_auc = average_precision_score(y_true, probabilities)
    brier = brier_score_loss(y_true, probabilities)

    precision, recall, thresholds = precision_recall_curve(
        y_true, probabilities
    )

    f1_scores = (
        2 * precision * recall
        / (precision + recall + 1e-10)
    )

    best_idx = int(np.argmax(f1_scores))

    if len(thresholds) == 0:
        best_threshold = 0.5
    elif best_idx < len(thresholds):
        best_threshold = thresholds[best_idx]
    else:
        best_threshold = thresholds[-1]

    # Find threshold closest to 0.50.
    if len(thresholds):
        standard_idx = int(
            np.argmin(np.abs(thresholds - 0.50))
        )
        standard_threshold = thresholds[standard_idx]
    else:
        standard_idx = 0
        standard_threshold = 0.50

    # Highest threshold that still gives >= 90% recall.
    recall_candidates = np.where(recall[:-1] >= 0.90)[0]

    if len(recall_candidates):
        hr_idx = int(recall_candidates[-1])
        hr_threshold = thresholds[hr_idx]
        hr_precision = precision[hr_idx]
        hr_recall = recall[hr_idx]
    else:
        hr_idx = None
        hr_threshold = np.nan
        hr_precision = np.nan
        hr_recall = np.nan

    best_pred = (
        probabilities >= best_threshold
    ).astype(int)

    return {
        "Model": model_name,
        "Split": split_name,
        "Feature_Set": feature_set,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Brier_Score": brier,
        "Optimal_Threshold": best_threshold,
        "Precision_at_F1opt": precision[best_idx],
        "Recall_at_F1opt": recall[best_idx],
        "F1_Score": f1_scores[best_idx],
        "90Recall_Threshold": hr_threshold,
        "90Recall_Precision": hr_precision,
        "90Recall_Recall": hr_recall,
        "Accuracy_at_F1opt": accuracy_score(
            y_true, best_pred
        ),
        "Standard_0.50_F1": threshold_metrics(
            y_true, probabilities, 0.50
        )["F1"],
        "Standard_0.50_Precision": threshold_metrics(
            y_true, probabilities, 0.50
        )["Precision"],
        "Standard_0.50_Recall": threshold_metrics(
            y_true, probabilities, 0.50
        )["Recall"],
    }


def print_evaluation(result):
    print(
        f"{result['Model']:<24} "
        f"ROC-AUC={result['ROC-AUC']:.4f} | "
        f"PR-AUC={result['PR-AUC']:.4f} | "
        f"Brier={result['Brier_Score']:.6f} | "
        f"F1={result['F1_Score']:.4f}"
    )


# ============================================================
# 5B. PRESENTATION / DIAGNOSTIC VISUAL HELPERS
# ============================================================

def save_confusion_matrix_plot(y_true, probabilities, threshold, title, filename):
    """Save a notebook-style confusion matrix at an explicit threshold."""
    if not RUN_PLOTS or not RUN_NOTEBOOK_VISUALS:
        return

    preds = (probabilities >= threshold).astype(int)
    cm = confusion_matrix(y_true, preds)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "Not Seriously Delinquent",
            "Seriously Delinquent",
        ],
    )
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    disp.plot(ax=ax, values_format=",d", cmap="Blues", colorbar=False)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=160)
    plt.close(fig)


def save_roc_plot(y_true, probabilities, title, filename, label):
    """Save a standalone ROC curve matching the original RF notebook output."""
    if not RUN_PLOTS or not RUN_NOTEBOOK_VISUALS:
        return

    fpr, tpr, _ = roc_curve(y_true, probabilities)
    auc = roc_auc_score(y_true, probabilities)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fpr, tpr, label=f"{label} AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=160)
    plt.close(fig)


def save_training_history_plot(model, title_prefix, filename):
    """Save AUC/PR-AUC and loss curves from a Keras training run."""
    if not RUN_PLOTS or not RUN_NOTEBOOK_VISUALS:
        return

    history = getattr(model, "_training_history", None)
    if history is None:
        return

    hist = history.history
    metric_key = "auc" if "auc" in hist else "pr_auc" if "pr_auc" in hist else None
    val_metric_key = f"val_{metric_key}" if metric_key else None

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    if metric_key and val_metric_key in hist:
        axes[0].plot(hist[metric_key], label=f"Train {metric_key.upper()}")
        axes[0].plot(hist[val_metric_key], label=f"Validation {metric_key.upper()}")
        axes[0].set_title(f"{title_prefix}: Model Performance")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel(metric_key.upper())
        axes[0].legend()

    axes[1].plot(hist.get("loss", []), label="Train Loss")
    axes[1].plot(hist.get("val_loss", []), label="Validation Loss")
    axes[1].set_title(f"{title_prefix}: Model Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Binary Cross-Entropy Loss")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=160)
    plt.close(fig)


def save_strict_threshold_report(y_val, val_probs, y_test, test_probs, model_name, split_type):
    """Choose threshold on validation data, then evaluate once on untouched test data."""
    if not RUN_STRICT_THRESHOLD_REPORT:
        return None

    precision, recall, thresholds = precision_recall_curve(y_val, val_probs)
    if len(thresholds) == 0:
        threshold = 0.50
    else:
        f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(
            precision[:-1] + recall[:-1], 1e-12
        )
        threshold = float(thresholds[int(np.nanargmax(f1_values))])

    metrics = threshold_metrics(y_test, test_probs, threshold)
    metrics.update({
        "Model": model_name,
        "Split": split_type,
        "Threshold_Selected_On": "Validation",
        "Validation_Optimal_Threshold": threshold,
    })
    return metrics


def save_data_quality_report(df):
    """Produce a compact data-quality report for presentation/reproducibility."""
    if not RUN_DATA_QUALITY_REPORT:
        return

    report = {
        "Rows": len(df),
        "Unique_Loans": df["loan_sequence_number"].nunique(),
        "Positive_Target_Rate": df["next_month_serious_delinquency"].mean(),
        "Missing_Target": int(df["next_month_serious_delinquency"].isna().sum()),
        "Irregular_Month_Transitions": int((df["month_diff_days"] > 32).sum()),
        "Missing_Normalized_UPB": int(df["normalized_upb"].isna().sum()),
        "Missing_UPB_Momentum": int(df["upb_pct_change_1m"].isna().sum()),
        "Missing_Credit_Score": int(df["credit_score"].isna().sum()) if "credit_score" in df else 0,
    }
    pd.DataFrame([report]).to_csv(
        RESULTS_DIR / "data_quality_report.csv", index=False
    )
    print("\nData-quality report:")
    print(pd.Series(report).to_string())


# ============================================================
# 6. RAW TABULAR RANDOM FOREST
# ============================================================

def run_raw_random_forest(df):
    print("\n" + "=" * 80)
    print("3. RAW/TABULAR RANDOM FOREST BASELINE")
    print("=" * 80)

    available = [c for c in RAW_FEATURES if c in df.columns]

    model_df = df[
        ["loan_sequence_number", "next_month_serious_delinquency"]
        + available
    ].copy()

    X = model_df[available]
    y = model_df["next_month_serious_delinquency"]
    groups = model_df["loan_sequence_number"]

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=RANDOM_STATE,
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups=groups)
    )

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    model = RandomForestClassifier(
        n_estimators=RF_TREES,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.50).astype(int)

    result = {
        "Model": "Raw Random Forest",
        "Split": "LOAN_LEVEL",
        "Feature_Set": "Raw_17",
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(
            y_test, preds, zero_division=0
        ),
        "Recall": recall_score(
            y_test, preds, zero_division=0
        ),
        "F1": f1_score(
            y_test, preds, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probs),
        "PR-AUC": average_precision_score(y_test, probs),
    }

    print(pd.Series(result).to_string())

    importance = pd.DataFrame({
        "Feature": available,
        "Importance": model.feature_importances_,
    }).sort_values(
        "Importance", ascending=False
    )

    importance.to_csv(
        RESULTS_DIR / "raw_random_forest_feature_importance.csv",
        index=False,
    )

    if RUN_PLOTS:
        plt.figure(figsize=(9, 6))
        plt.barh(
            importance["Feature"].iloc[::-1],
            importance["Importance"].iloc[::-1],
        )
        plt.xlabel("Importance")
        plt.ylabel("Feature")
        plt.title("Raw Random Forest Feature Importance")
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR / "raw_random_forest_feature_importance.png",
            dpi=150,
        )
        plt.close()

        save_confusion_matrix_plot(
            y_test, probs, 0.50,
            "Random Forest Confusion Matrix",
            "raw_random_forest_confusion_matrix.png",
        )
        save_roc_plot(
            y_test, probs,
            "Random Forest ROC Curve",
            "raw_random_forest_roc_curve.png",
            "Random Forest",
        )

    return result, model, probs, y_test


# ============================================================
# 7. SEQUENTIAL RANDOM FOREST + THRESHOLD + SMOTE
# ============================================================

def run_sequence_random_forest(df):
    print("\n" + "=" * 80)
    print("4. 12-MONTH SEQUENTIAL RANDOM FOREST")
    print("=" * 80)

    features = [
        "current_actual_upb",
        "current_interest_rate",
        "loan_age",
        "remaining_months_to_legal_maturity",
        "credit_score",
        "estimated_loan_to_value",
        "changeDelinquency",
        "interest_bearing_upb",
        "current_deferred_upb",
        "original_loan_to_value",
        "original_debt_to_income_ratio",
        "original_interest_rate",
        "number_of_borrowers",
        "mortgage_insurance_percentage",
    ]

    features = [c for c in features if c in df.columns]

    data = prepare_data(
        df,
        features,
        "LOAN_LEVEL",
        SEQUENCE_LENGTH,
    )

    X_train = data["X_train_2d"]
    X_val = data["X_val_2d"]
    X_test = data["X_test_2d"]
    y_train = data["y_train"]
    y_test = data["y_test"]

    # Baseline RF.
    rf = RandomForestClassifier(
        n_estimators=SEQUENCE_RF_TREES,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)

    baseline_probs = rf.predict_proba(X_test)[:, 1]

    baseline_result = evaluate_model(
        y_test,
        baseline_probs,
        "Sequential Random Forest",
        "LOAN_LEVEL",
        "14_raw_sequential",
    )

    print_evaluation(baseline_result)

    # Threshold analysis.
    threshold_values = [
        0.50, 0.30, 0.20, 0.15, 0.10, 0.05
    ]

    threshold_rows = []
    for threshold in threshold_values:
        row = threshold_metrics(
            y_test, baseline_probs, threshold
        )
        threshold_rows.append(row)

    threshold_df = pd.DataFrame(threshold_rows)
    threshold_df.to_csv(
        RESULTS_DIR / "sequential_rf_thresholds.csv",
        index=False,
    )

    print("\nSequential RF threshold analysis:")
    print(threshold_df.round(4).to_string(index=False))

    # SMOTE is applied only to training data.
    smote_result = None

    if SMOTE is not None:
        print("\nTraining Random Forest + SMOTE...")
        smote = SMOTE(
            random_state=RANDOM_STATE,
            k_neighbors=3,
        )

        X_train_smote, y_train_smote = smote.fit_resample(
            X_train, y_train
        )

        rf_smote = RandomForestClassifier(
            n_estimators=SEQUENCE_RF_TREES,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        rf_smote.fit(X_train_smote, y_train_smote)

        smote_probs = rf_smote.predict_proba(
            X_test
        )[:, 1]

        smote_result = evaluate_model(
            y_test,
            smote_probs,
            "Sequential RF + SMOTE",
            "LOAN_LEVEL",
            "14_raw_sequential",
        )

        print_evaluation(smote_result)

        smote_threshold_rows = []
        for threshold in threshold_values:
            smote_threshold_rows.append(
                threshold_metrics(
                    y_test,
                    smote_probs,
                    threshold,
                )
            )

        pd.DataFrame(
            smote_threshold_rows
        ).to_csv(
            RESULTS_DIR / "sequential_rf_smote_thresholds.csv",
            index=False,
        )

    else:
        print(
            "imblearn is not installed; skipping SMOTE."
        )

    if RUN_PLOTS and RUN_NOTEBOOK_VISUALS:
        # Original notebook threshold visuals.
        save_confusion_matrix_plot(
            y_test, baseline_probs, 0.50,
            "Baseline Random Forest - Threshold = 0.50",
            "sequential_rf_confusion_threshold_050.png",
        )
        save_confusion_matrix_plot(
            y_test, baseline_probs, 0.10,
            "Baseline Random Forest - Threshold = 0.10",
            "sequential_rf_confusion_threshold_010.png",
        )
        save_roc_plot(
            y_test, baseline_probs,
            "Baseline Random Forest ROC Curve",
            "sequential_rf_roc_curve.png",
            "Baseline Random Forest",
        )

        # Aggregate importance across the 12 lagged copies of each variable.
        importances = rf.feature_importances_.reshape(
            SEQUENCE_LENGTH, len(features)
        ).sum(axis=0)
        seq_importance = pd.DataFrame({
            "Feature": features,
            "Aggregated_Importance": importances,
        }).sort_values("Aggregated_Importance", ascending=False)
        seq_importance.to_csv(
            RESULTS_DIR / "sequential_rf_feature_importance.csv",
            index=False,
        )

        plt.figure(figsize=(9, 6))
        plt.barh(
            seq_importance["Feature"].iloc[::-1],
            seq_importance["Aggregated_Importance"].iloc[::-1],
        )
        plt.xlabel("Aggregated Importance Across 12 Months")
        plt.ylabel("Feature")
        plt.title("Sequential Random Forest Feature Importance")
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR / "sequential_rf_feature_importance.png",
            dpi=160,
        )
        plt.close()

    return {
        "baseline": baseline_result,
        "smote": smote_result,
        "thresholds": threshold_df,
        "data": data,
    }


# ============================================================
# 8. LOGISTIC REGRESSION
# ============================================================

def run_logistic_models(df, split_type):
    print("\n" + "=" * 80)
    print(
        f"5. LOGISTIC REGRESSION EXPERIMENTS - {split_type}"
    )
    print("=" * 80)

    rows = []
    prediction_store = {}

    # Feature ablation.
    for set_name, features in FEATURE_SETS.items():
        print(f"\nFeature set: {set_name}")

        data = prepare_data(
            df,
            features,
            split_type,
            SEQUENCE_LENGTH,
        )

        X_train = data["X_train_2d"]
        X_test = data["X_test_2d"]
        y_train = data["y_train"]
        y_test = data["y_test"]

        model = LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
        )

        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:, 1]

        result = evaluate_model(
            y_test,
            probs,
            "Logistic Regression",
            split_type,
            set_name,
        )

        rows.append(result)
        prediction_store[
            f"plain_{set_name}"
        ] = (probs, y_test)

        print_evaluation(result)

    ablation_df = pd.DataFrame(rows)
    ablation_df.to_csv(
        RESULTS_DIR
        / f"logistic_feature_ablation_{split_type}.csv",
        index=False,
    )

    # Imbalance comparison on D_Full.
    features = FEATURE_SETS["D_Full"]
    data = prepare_data(
        df,
        features,
        split_type,
        SEQUENCE_LENGTH,
    )

    X_train = data["X_train_2d"]
    X_test = data["X_test_2d"]
    y_train = data["y_train"]
    y_test = data["y_test"]

    imbalance_rows = []

    imbalance_models = {
        "Plain LR": LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
        ),
        "LR class_weight=balanced": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
    }

    for name, model in imbalance_models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:, 1]

        result = evaluate_model(
            y_test,
            probs,
            name,
            split_type,
            "D_Full",
        )

        imbalance_rows.append(result)
        prediction_store[
            name
        ] = (probs, y_test)

        print_evaluation(result)

    # SMOTE logistic regression.
    if SMOTE is not None:
        print("\nTraining LR + SMOTE...")
        smote = SMOTE(
            random_state=RANDOM_STATE,
            k_neighbors=3,
        )

        X_train_smote, y_train_smote = smote.fit_resample(
            X_train, y_train
        )

        smote_lr = LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
        )

        smote_lr.fit(
            X_train_smote,
            y_train_smote,
        )

        probs = smote_lr.predict_proba(X_test)[:, 1]

        result = evaluate_model(
            y_test,
            probs,
            "LR + SMOTE",
            split_type,
            "D_Full",
        )

        imbalance_rows.append(result)
        prediction_store[
            "smote"
        ] = (probs, y_test)

        print_evaluation(result)

    imbalance_df = pd.DataFrame(imbalance_rows)
    imbalance_df.to_csv(
        RESULTS_DIR
        / f"logistic_imbalance_{split_type}.csv",
        index=False,
    )

    # ROC and PR curves.
    if RUN_PLOTS and split_type == "OOT":
        plt.figure(figsize=(7, 6))

        for key, (probs, labels) in prediction_store.items():
            if key in ["plain_D_Full", "LR class_weight=balanced", "smote"]:
                fpr, tpr, _ = roc_curve(
                    labels, probs
                )
                plt.plot(
                    fpr,
                    tpr,
                    label=key,
                )

        plt.plot(
            [0, 1], [0, 1],
            linestyle="--",
        )
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("Logistic Regression ROC Curves - OOT")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR / "logistic_roc_oot.png",
            dpi=150,
        )
        plt.close()

        plt.figure(figsize=(7, 6))

        for key, (probs, labels) in prediction_store.items():
            if key in ["plain_D_Full", "LR class_weight=balanced", "smote"]:
                precision, recall, _ = precision_recall_curve(
                    labels, probs
                )
                plt.plot(
                    recall,
                    precision,
                    label=key,
                )

        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(
            "Logistic Regression Precision-Recall Curves - OOT"
        )
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR / "logistic_pr_oot.png",
            dpi=150,
        )
        plt.close()

    return ablation_df, imbalance_df


def run_logistic_regularization_sweep(df, split_type):
    print("\n" + "=" * 80)
    print(
        f"6. LOGISTIC REGULARIZATION SWEEP - {split_type}"
    )
    print("=" * 80)

    features = FEATURE_SETS["D_Full"]

    data = prepare_data(
        df,
        features,
        split_type,
        SEQUENCE_LENGTH,
    )

    X_train = data["X_train_2d"]
    X_val = data["X_val_2d"]
    y_train = data["y_train"]
    y_val = data["y_val"]

    C_values = [
        0.001, 0.01, 0.1,
        1.0, 10.0, 100.0,
    ]

    rows = []

    for C in C_values:
        start = time.time()

        model = LogisticRegression(
            C=C,
            max_iter=1000,
            random_state=RANDOM_STATE,
        )

        model.fit(X_train, y_train)
        probs = model.predict_proba(X_val)[:, 1]

        rows.append({
            "C": C,
            "n_iter": int(
                np.max(model.n_iter_)
            ),
            "fit_seconds": time.time() - start,
            "ROC-AUC": roc_auc_score(
                y_val, probs
            ),
            "PR-AUC": average_precision_score(
                y_val, probs
            ),
        })

    result_df = pd.DataFrame(rows)

    result_df.to_csv(
        RESULTS_DIR
        / f"logistic_regularization_{split_type}.csv",
        index=False,
    )

    print(result_df.round(5).to_string(index=False))

    if RUN_PLOTS:
        plt.figure(figsize=(8, 5))
        plt.plot(
            result_df["C"],
            result_df["ROC-AUC"],
            marker="o",
            label="ROC-AUC",
        )
        plt.plot(
            result_df["C"],
            result_df["PR-AUC"],
            marker="s",
            label="PR-AUC",
        )
        plt.xscale("log")
        plt.xlabel("C (inverse regularization strength)")
        plt.ylabel("Score")
        plt.title(
            f"Logistic Regularization - {split_type}"
        )
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR
            / f"logistic_regularization_{split_type}.png",
            dpi=150,
        )
        plt.close()

    return result_df


# ============================================================
# 9. DEEP LEARNING MODEL BUILDERS
# ============================================================

def require_tensorflow():
    if tf is None:
        raise ImportError(
            "TensorFlow is required for GRU/LSTM experiments. "
            "Install tensorflow or set RUN_FULL_SHOOTOUT=False."
        )


def build_gru(input_shape):
    require_tensorflow()

    model = Sequential([
        GRU(
            64,
            input_shape=input_shape,
        ),
        Dropout(0.30),
        Dense(32, activation="relu"),
        Dropout(0.20),
        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR",
            )
        ],
    )

    return model


def build_basic_lstm(input_shape):
    """Basic 64-unit LSTM from the original baseline notebook."""
    require_tensorflow()

    model = Sequential([
        LSTM(64, input_shape=input_shape, return_sequences=False),
        Dropout(0.20),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )
    return model


def build_custom_lstm(input_shape):
    require_tensorflow()

    model = Sequential([
        LSTM(
            64,
            input_shape=input_shape,
            return_sequences=True,
        ),
        Dropout(0.30),
        LSTM(
            32,
            return_sequences=False,
        ),
        BatchNormalization(),
        Dropout(0.20),
        Dense(
            32,
            kernel_initializer="he_normal",
        ),
        LeakyReLU(negative_slope=0.01),
        Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR",
            )
        ],
    )

    return model


def fit_dl_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=DL_EPOCHS,
):
    callbacks = [
        EarlyStopping(
            monitor="val_pr_auc",
            patience=4,
            mode="max",
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(
            monitor="val_pr_auc",
            factor=0.2,
            patience=2,
            min_lr=0.0001,
        ),
    ]

    history = model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=DL_BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=0,
    )

    # Keep the History object attached so the original notebook-style
    # training curves can be reproduced after the model is fit.
    model._training_history = history
    return model


# ============================================================
# 10. FULL MODEL SHOOTOUT
# ============================================================

def run_full_model_shootout(df, split_type):
    require_tensorflow()

    if XGBClassifier is None:
        print("XGBoost not installed; XGBoost will be skipped.")

    if LGBMClassifier is None:
        print("LightGBM not installed; LightGBM will be skipped.")

    print("\n" + "=" * 80)
    print(
        f"7. FULL MODEL SHOOTOUT - {split_type}"
    )
    print("=" * 80)

    features = FEATURE_SETS["D_Full"]

    data = prepare_data(
        df,
        features,
        split_type,
        SEQUENCE_LENGTH,
    )

    X_train_3d = data["X_train_3d"]
    X_val_3d = data["X_val_3d"]
    X_test_3d = data["X_test_3d"]

    X_train_2d = data["X_train_2d"]
    X_test_2d = data["X_test_2d"]

    y_train = data["y_train"]
    y_val = data["y_val"]
    y_test = data["y_test"]

    results = []
    probability_store = {}

    # Naive baseline.
    naive_probs = np.zeros_like(
        y_test,
        dtype=float,
    )

    naive_result = evaluate_model(
        y_test,
        naive_probs,
        "Naive All-0s",
        split_type,
        "D_Full",
    )

    results.append(naive_result)
    probability_store["Naive All-0s"] = naive_probs
    print_evaluation(naive_result)

    # Logistic Regression.
    print("\nTraining Logistic Regression...")
    lr = LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    lr.fit(X_train_2d, y_train)

    lr_probs = lr.predict_proba(X_test_2d)[:, 1]

    lr_result = evaluate_model(
        y_test,
        lr_probs,
        "Logistic Regression",
        split_type,
        "D_Full",
    )

    results.append(lr_result)
    probability_store["Logistic Regression"] = lr_probs
    print_evaluation(lr_result)

    # Random Forest.
    print("\nTraining Random Forest...")
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf.fit(X_train_2d, y_train)

    rf_probs = rf.predict_proba(X_test_2d)[:, 1]

    rf_result = evaluate_model(
        y_test,
        rf_probs,
        "Random Forest",
        split_type,
        "D_Full",
    )

    results.append(rf_result)
    probability_store["Random Forest"] = rf_probs
    print_evaluation(rf_result)

    # XGBoost.
    if XGBClassifier is not None:
        print("\nTraining XGBoost...")
        xgb = XGBClassifier(
            n_estimators=XGB_TREES,
            max_depth=6,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            n_jobs=-1,
        )

        xgb.fit(X_train_2d, y_train)

        xgb_probs = xgb.predict_proba(
            X_test_2d
        )[:, 1]

        xgb_result = evaluate_model(
            y_test,
            xgb_probs,
            "XGBoost",
            split_type,
            "D_Full",
        )

        results.append(xgb_result)
        probability_store["XGBoost"] = xgb_probs
        print_evaluation(xgb_result)

    # LightGBM.
    if LGBMClassifier is not None:
        print("\nTraining LightGBM...")
        lgbm = LGBMClassifier(
            n_estimators=LGBM_TREES,
            max_depth=6,
            random_state=RANDOM_STATE,
            verbosity=-1,
        )

        lgbm.fit(X_train_2d, y_train)

        lgbm_probs = lgbm.predict_proba(
            X_test_2d
        )[:, 1]

        lgbm_result = evaluate_model(
            y_test,
            lgbm_probs,
            "LightGBM",
            split_type,
            "D_Full",
        )

        results.append(lgbm_result)
        probability_store["LightGBM"] = lgbm_probs
        print_evaluation(lgbm_result)

    # GRU.
    print("\nTraining GRU...")
    gru = build_gru(
        (
            X_train_3d.shape[1],
            X_train_3d.shape[2],
        )
    )

    gru = fit_dl_model(
        gru,
        X_train_3d,
        y_train,
        X_val_3d,
        y_val,
    )

    gru_probs = gru.predict(
        X_test_3d,
        batch_size=DL_BATCH_SIZE,
        verbose=0,
    ).ravel()

    gru_result = evaluate_model(
        y_test,
        gru_probs,
        "Basic GRU",
        split_type,
        "D_Full",
    )

    results.append(gru_result)
    probability_store["Basic GRU"] = gru_probs
    print_evaluation(gru_result)
    save_training_history_plot(
        gru,
        f"Basic GRU ({split_type})",
        f"training_history_gru_{split_type}.png",
    )

    # Basic LSTM from the original class/project notebook.
    print("\nTraining Basic LSTM...")
    basic_lstm = build_basic_lstm(
        (X_train_3d.shape[1], X_train_3d.shape[2])
    )
    basic_lstm = fit_dl_model(
        basic_lstm,
        X_train_3d,
        y_train,
        X_val_3d,
        y_val,
    )
    basic_lstm_probs = basic_lstm.predict(
        X_test_3d,
        batch_size=DL_BATCH_SIZE,
        verbose=0,
    ).ravel()
    basic_lstm_result = evaluate_model(
        y_test,
        basic_lstm_probs,
        "Basic LSTM",
        split_type,
        "D_Full",
    )
    results.append(basic_lstm_result)
    probability_store["Basic LSTM"] = basic_lstm_probs
    print_evaluation(basic_lstm_result)
    save_training_history_plot(
        basic_lstm,
        f"Basic LSTM ({split_type})",
        f"training_history_basic_lstm_{split_type}.png",
    )
    save_confusion_matrix_plot(
        y_test, basic_lstm_probs, 0.50,
        f"Basic LSTM Confusion Matrix - {split_type}",
        f"basic_lstm_confusion_matrix_{split_type}.png",
    )

    # Custom LSTM.
    print("\nTraining Custom LSTM...")
    lstm = build_custom_lstm(
        (
            X_train_3d.shape[1],
            X_train_3d.shape[2],
        )
    )

    lstm = fit_dl_model(
        lstm,
        X_train_3d,
        y_train,
        X_val_3d,
        y_val,
    )

    lstm_probs = lstm.predict(
        X_test_3d,
        batch_size=DL_BATCH_SIZE,
        verbose=0,
    ).ravel()

    lstm_result = evaluate_model(
        y_test,
        lstm_probs,
        "Custom LSTM",
        split_type,
        "D_Full",
    )

    results.append(lstm_result)
    probability_store["Custom LSTM"] = lstm_probs
    print_evaluation(lstm_result)
    save_training_history_plot(
        lstm,
        f"Custom LSTM ({split_type})",
        f"training_history_lstm_{split_type}.png",
    )
    save_confusion_matrix_plot(
        y_test, lstm_probs, 0.50,
        f"LSTM Confusion Matrix - {split_type}",
        f"lstm_confusion_matrix_{split_type}.png",
    )

    # Strict threshold evaluation: threshold is selected on validation data,
    # never on the final test set. This is a methodological improvement over
    # the exploratory threshold optimization used in some original cells.
    if RUN_STRICT_THRESHOLD_REPORT:
        strict_rows = []
        strict_inputs = [
            ("Logistic Regression", lr_probs),
            ("Random Forest", rf_probs),
        ]
        if "XGBoost" in probability_store:
            strict_inputs.append(("XGBoost", probability_store["XGBoost"]))
        if "LightGBM" in probability_store:
            strict_inputs.append(("LightGBM", probability_store["LightGBM"]))
        strict_inputs.extend([
            ("Basic GRU", gru_probs),
            ("Basic LSTM", basic_lstm_probs),
            ("Custom LSTM", lstm_probs),
        ])

        # Refit/predict validation probabilities for the models that expose
        # their fitted objects above. DL validation predictions are available
        # directly from the fitted networks.
        validation_probs = {
            "Logistic Regression": lr.predict_proba(data["X_val_2d"])[:, 1],
            "Random Forest": rf.predict_proba(data["X_val_2d"])[:, 1],
            "Basic GRU": gru.predict(data["X_val_3d"], batch_size=DL_BATCH_SIZE, verbose=0).ravel(),
            "Basic LSTM": basic_lstm.predict(data["X_val_3d"], batch_size=DL_BATCH_SIZE, verbose=0).ravel(),
            "Custom LSTM": lstm.predict(data["X_val_3d"], batch_size=DL_BATCH_SIZE, verbose=0).ravel(),
        }
        if "XGBoost" in probability_store:
            validation_probs["XGBoost"] = xgb.predict_proba(data["X_val_2d"])[:, 1]
        if "LightGBM" in probability_store:
            validation_probs["LightGBM"] = lgbm.predict_proba(data["X_val_2d"])[:, 1]

        for model_name, test_probs in strict_inputs:
            row = save_strict_threshold_report(
                data["y_val"],
                validation_probs[model_name],
                y_test,
                test_probs,
                model_name,
                split_type,
            )
            if row is not None:
                strict_rows.append(row)

        if strict_rows:
            pd.DataFrame(strict_rows).to_csv(
                RESULTS_DIR / f"strict_validation_thresholds_{split_type}.csv",
                index=False,
            )

    result_df = pd.DataFrame(results)

    result_df.to_csv(
        RESULTS_DIR
        / f"full_model_shootout_{split_type}.csv",
        index=False,
    )

    # Calibration plot.
    if RUN_PLOTS:
        plt.figure(figsize=(9, 7))
        plt.plot(
            [0, 1],
            [0, 1],
            linestyle=":",
            label="Perfect calibration",
        )

        for name, probs in probability_store.items():
            if name == "Naive All-0s":
                continue

            frac_pos, mean_pred = calibration_curve(
                y_test,
                probs,
                n_bins=10,
                strategy="quantile",
            )

            plt.plot(
                mean_pred,
                frac_pos,
                marker="s",
                label=name,
            )

        plt.xlabel(
            "Mean Predicted Probability"
        )
        plt.ylabel(
            "Fraction of Actual Positives"
        )
        plt.title(
            f"Calibration Curves - {split_type}"
        )
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR
            / f"calibration_{split_type}.png",
            dpi=150,
        )
        plt.close()

        # PR curves.
        plt.figure(figsize=(8, 7))

        for name, probs in probability_store.items():
            if name == "Naive All-0s":
                continue

            precision, recall, _ = precision_recall_curve(
                y_test,
                probs,
            )

            plt.plot(
                recall,
                precision,
                label=name,
            )

        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(
            f"Precision-Recall Curves - {split_type}"
        )
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR
            / f"precision_recall_{split_type}.png",
            dpi=150,
        )
        plt.close()

    return result_df


# ============================================================
# 11. FEATURE ABLATION: XGBOOST + LSTM
# ============================================================

def run_feature_ablation(df):
    require_tensorflow()

    if XGBClassifier is None:
        print(
            "XGBoost not installed; feature ablation skipped."
        )
        return pd.DataFrame()

    print("\n" + "=" * 80)
    print("8. FEATURE ABLATION - OOT")
    print("=" * 80)

    rows = []

    for set_name, features in FEATURE_SETS.items():
        print(f"\nTesting {set_name}...")

        data = prepare_data(
            df,
            features,
            "OOT",
            SEQUENCE_LENGTH,
        )

        X_train_3d = data["X_train_3d"]
        X_val_3d = data["X_val_3d"]
        X_test_3d = data["X_test_3d"]

        X_train_2d = data["X_train_2d"]
        X_test_2d = data["X_test_2d"]

        y_train = data["y_train"]
        y_val = data["y_val"]
        y_test = data["y_test"]

        # XGBoost.
        xgb = XGBClassifier(
            n_estimators=XGB_TREES,
            max_depth=6,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            n_jobs=-1,
        )

        xgb.fit(X_train_2d, y_train)

        xgb_probs = xgb.predict_proba(
            X_test_2d
        )[:, 1]

        rows.append(
            evaluate_model(
                y_test,
                xgb_probs,
                "XGBoost",
                "OOT",
                set_name,
            )
        )

        # LSTM.
        lstm = build_custom_lstm(
            (
                X_train_3d.shape[1],
                X_train_3d.shape[2],
            )
        )

        lstm = fit_dl_model(
            lstm,
            X_train_3d,
            y_train,
            X_val_3d,
            y_val,
            epochs=ABLATION_DL_EPOCHS,
        )

        lstm_probs = lstm.predict(
            X_test_3d,
            batch_size=DL_BATCH_SIZE,
            verbose=0,
        ).ravel()

        rows.append(
            evaluate_model(
                y_test,
                lstm_probs,
                "Class-LSTM",
                "OOT",
                set_name,
            )
        )

    result_df = pd.DataFrame(rows)

    result_df.to_csv(
        RESULTS_DIR / "feature_ablation_oot.csv",
        index=False,
    )

    print("\nFeature ablation results:")
    print(
        result_df.round(4).to_string(index=False)
    )

    if RUN_PLOTS:
        pivot = result_df.pivot(
            index="Feature_Set",
            columns="Model",
            values="PR-AUC",
        )

        pivot.plot(
            kind="bar",
            figsize=(9, 6),
        )

        plt.ylabel("PR-AUC")
        plt.title(
            "Feature Ablation: PR-AUC by Feature Set"
        )
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR
            / "feature_ablation_pr_auc.png",
            dpi=150,
        )
        plt.close()

    return result_df


# ============================================================
# 12. TEMPORAL SENSITIVITY
# ============================================================

def run_temporal_sensitivity(df):
    require_tensorflow()

    print("\n" + "=" * 80)
    print("9. TEMPORAL SENSITIVITY - LSTM")
    print("=" * 80)

    rows = []

    features = FEATURE_SETS["D_Full"]

    for seq_len in TEMPORAL_SEQUENCE_LENGTHS:
        print(
            f"\nSequence length: {seq_len} months"
        )

        start = time.time()

        data = prepare_data(
            df,
            features,
            "OOT",
            seq_len,
        )

        X_train = data["X_train_3d"]
        X_val = data["X_val_3d"]
        X_test = data["X_test_3d"]

        y_train = data["y_train"]
        y_val = data["y_val"]
        y_test = data["y_test"]

        model = Sequential([
            LSTM(
                64,
                input_shape=(
                    X_train.shape[1],
                    X_train.shape[2],
                ),
                return_sequences=False,
            ),
            Dropout(0.30),
            Dense(32, activation="relu"),
            Dropout(0.20),
            Dense(1, activation="sigmoid"),
        ])

        model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=[
                tf.keras.metrics.AUC(
                    name="pr_auc",
                    curve="PR",
                )
            ],
        )

        model = fit_dl_model(
            model,
            X_train,
            y_train,
            X_val,
            y_val,
        )

        probs = model.predict(
            X_test,
            batch_size=DL_BATCH_SIZE,
            verbose=0,
        ).ravel()

        elapsed = time.time() - start

        rows.append({
            "Sequence_Length": seq_len,
            "ROC-AUC": roc_auc_score(
                y_test, probs
            ),
            "PR-AUC": average_precision_score(
                y_test, probs
            ),
            "Brier_Score": brier_score_loss(
                y_test, probs
            ),
            "Execution_Time_Sec": round(
                elapsed, 1
            ),
        })

        print(
            f"PR-AUC={rows[-1]['PR-AUC']:.4f} | "
            f"Time={elapsed:.1f}s"
        )

    result_df = pd.DataFrame(rows)

    result_df.to_csv(
        RESULTS_DIR / "temporal_sensitivity.csv",
        index=False,
    )

    print("\nTemporal sensitivity:")
    print(
        result_df.round(4).to_string(index=False)
    )

    if RUN_PLOTS:
        plt.figure(figsize=(8, 5))
        plt.plot(
            result_df["Sequence_Length"],
            result_df["PR-AUC"],
            marker="o",
        )
        plt.xlabel("Lookback Window (Months)")
        plt.ylabel("PR-AUC")
        plt.title(
            "LSTM Temporal Sensitivity"
        )
        plt.xticks(
            TEMPORAL_SEQUENCE_LENGTHS
        )
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR
            / "temporal_sensitivity_pr_auc.png",
            dpi=150,
        )
        plt.close()

        plt.figure(figsize=(8, 5))
        plt.plot(
            result_df["Sequence_Length"],
            result_df["Execution_Time_Sec"],
            marker="o",
        )
        plt.xlabel("Lookback Window (Months)")
        plt.ylabel("Execution Time (Seconds)")
        plt.title(
            "LSTM Computational Cost by Lookback"
        )
        plt.xticks(
            TEMPORAL_SEQUENCE_LENGTHS
        )
        plt.tight_layout()
        plt.savefig(
            RESULTS_DIR
            / "temporal_sensitivity_time.png",
            dpi=150,
        )
        plt.close()

    return result_df


# ============================================================
# 13. MASTER REPORT
# ============================================================

def build_master_report(all_results):
    print("\n" + "=" * 80)
    print("10. BUILDING MASTER RESULTS")
    print("=" * 80)

    frames = []

    for name, obj in all_results.items():
        if isinstance(obj, pd.DataFrame) and not obj.empty:
            temp = obj.copy()
            temp["Experiment"] = name
            frames.append(temp)

        elif isinstance(obj, dict):
            # Convert nested result dictionaries when possible.
            for sub_name, sub_obj in obj.items():
                if isinstance(sub_obj, pd.DataFrame):
                    temp = sub_obj.copy()
                    temp["Experiment"] = (
                        f"{name}_{sub_name}"
                    )
                    frames.append(temp)

    if frames:
        master = pd.concat(
            frames,
            ignore_index=True,
            sort=False,
        )
    else:
        master = pd.DataFrame()

    master.to_csv(
        RESULTS_DIR / "MASTER_RESULTS.csv",
        index=False,
    )

    # Presentation-ready model progression: OOT D_Full models only.
    if RUN_NOTEBOOK_VISUALS and not master.empty and {"Model", "Split", "Feature_Set", "PR-AUC"}.issubset(master.columns):
        progression_order = [
            "Logistic Regression",
            "Random Forest",
            "XGBoost",
            "LightGBM",
            "Basic GRU",
            "Basic LSTM",
            "Custom LSTM",
        ]
        prog = master[
            (master["Split"] == "OOT")
            & (master["Feature_Set"] == "D_Full")
            & master["Model"].isin(progression_order)
        ].copy()
        if not prog.empty:
            prog["Model"] = pd.Categorical(
                prog["Model"], categories=progression_order, ordered=True
            )
            prog = prog.sort_values("Model")
            prog.to_csv(RESULTS_DIR / "model_progression_oot.csv", index=False)
            ax = prog.plot(
                x="Model", y="PR-AUC", kind="bar", figsize=(10, 6), legend=False
            )
            ax.set_ylabel("PR-AUC")
            ax.set_xlabel("Model progression")
            ax.set_title("Progressive Model Comparison - OOT")
            plt.xticks(rotation=35, ha="right")
            plt.tight_layout()
            plt.savefig(RESULTS_DIR / "model_progression_oot.png", dpi=160)
            plt.close()

    # Human-readable inventory for the final presentation package.
    if RUN_NOTEBOOK_VISUALS:
        plot_files = sorted(
            str(p.relative_to(RESULTS_DIR))
            for p in RESULTS_DIR.glob("*.png")
        )
        pd.DataFrame({"Visualization": plot_files}).to_csv(
            RESULTS_DIR / "VISUALIZATION_INDEX.csv", index=False
        )

    print(
        f"Saved master results to: "
        f"{RESULTS_DIR / 'MASTER_RESULTS.csv'}"
    )

    return master


# ============================================================
# 14. MAIN
# ============================================================

def main():
    start_all = time.time()

    df = load_and_prepare_data(
        DATA_PATH,
        n_loans=N_LOANS,
    )

    df = engineer_temporal_features(df)

    # Save a compact modeling sample for reproducibility.
    df.head(10000).to_csv(
        RESULTS_DIR / "modeling_sample_preview.csv",
        index=False,
    )

    all_results = {}

    # --------------------------------------------------------
    # Raw RF from the first notebook.
    # --------------------------------------------------------
    if RUN_RAW_RF:
        try:
            raw_result, _, _, _ = run_raw_random_forest(df)
            all_results["Raw_RF"] = pd.DataFrame([raw_result])
        except Exception as e:
            print(
                f"\nRaw RF failed: {type(e).__name__}: {e}"
            )

    # --------------------------------------------------------
    # Sequential RF, threshold, SMOTE.
    # --------------------------------------------------------
    if RUN_SEQUENCE_RF:
        try:
            seq_results = run_sequence_random_forest(df)
            all_results["Sequential_RF"] = pd.DataFrame(
                [
                    seq_results["baseline"],
                    seq_results["smote"],
                ]
                if seq_results["smote"] is not None
                else [seq_results["baseline"]]
            )
        except Exception as e:
            print(
                f"\nSequential RF failed: "
                f"{type(e).__name__}: {e}"
            )

    # --------------------------------------------------------
    # Logistic regression experiments.
    # --------------------------------------------------------
    if RUN_LOGISTIC:
        for split in ["LOAN_LEVEL", "OOT"]:
            try:
                ablation, imbalance = run_logistic_models(
                    df, split
                )
                all_results[
                    f"Logistic_Ablation_{split}"
                ] = ablation
                all_results[
                    f"Logistic_Imbalance_{split}"
                ] = imbalance

                sweep = run_logistic_regularization_sweep(
                    df, split
                )
                all_results[
                    f"Logistic_Regularization_{split}"
                ] = sweep

            except Exception as e:
                print(
                    f"\nLogistic experiments failed for "
                    f"{split}: {type(e).__name__}: {e}"
                )

    # --------------------------------------------------------
    # Full model shootout.
    # --------------------------------------------------------
    if RUN_FULL_SHOOTOUT:
        for split in ["LOAN_LEVEL", "OOT"]:
            try:
                shootout = run_full_model_shootout(
                    df, split
                )
                all_results[
                    f"Full_Shootout_{split}"
                ] = shootout

            except Exception as e:
                print(
                    f"\nFull shootout failed for "
                    f"{split}: {type(e).__name__}: {e}"
                )

    # --------------------------------------------------------
    # Feature ablation.
    # --------------------------------------------------------
    if RUN_FEATURE_ABLATION:
        try:
            ablation = run_feature_ablation(df)
            all_results["Feature_Ablation_OOT"] = ablation
        except Exception as e:
            print(
                f"\nFeature ablation failed: "
                f"{type(e).__name__}: {e}"
            )

    # --------------------------------------------------------
    # Temporal sensitivity.
    # --------------------------------------------------------
    if RUN_TEMPORAL_SENSITIVITY:
        try:
            temporal = run_temporal_sensitivity(df)
            all_results[
                "Temporal_Sensitivity"
            ] = temporal
        except Exception as e:
            print(
                f"\nTemporal sensitivity failed: "
                f"{type(e).__name__}: {e}"
            )

    # --------------------------------------------------------
    # Master output.
    # --------------------------------------------------------
    master = build_master_report(all_results)

    print("\n" + "=" * 80)
    print("FINAL SUMMARY")
    print("=" * 80)

    if not master.empty:
        print(
            master.round(4).to_string(index=False)
        )

    print(
        f"\nTotal execution time: "
        f"{time.time() - start_all:.1f} seconds"
    )

    print(
        f"All CSVs and plots are in: "
        f"{RESULTS_DIR.resolve()}"
    )


if __name__ == "__main__":
    main()


Mounted at /content/drive
1. LOADING AND PREPARING DATA
Reading: /content/merged_mortgage_2013.csv
Original shape: (870258, 63)
Dropped 16 columns.
Working sample: 870,258 rows
Unique loans: 9,303

Target distribution:
next_month_serious_delinquency
0    857550
1      3219
Name: count, dtype: int64
next_month_serious_delinquency
0    99.626
1     0.374
Name: percent, dtype: float64

2. TEMPORAL FEATURE ENGINEERING
Irregular monthly transitions (>32 days): 0
Final positive rate: 0.3740%

Data-quality report:
Rows                           860769.00000
Unique_Loans                     9300.00000
Positive_Target_Rate                0.00374
Missing_Target                      0.00000
Irregular_Month_Transitions         0.00000
Missing_Normalized_UPB              0.00000
Missing_UPB_Momentum                0.00000
Missing_Credit_Score                0.00000

3. RAW/TABULAR RANDOM FOREST BASELINE
Model          Raw Random Forest
Split                 LOAN_LEVEL
Feature_Set               Raw_

In [6]:
# ================================================================
# FINAL VALIDATION / INTEGRITY AUDIT (STANDALONE CELL)
# ================================================================
import pandas as pd
import numpy as np

print("\n" + "=" * 80)
print("FINAL PIPELINE INTEGRITY AUDIT")
print("=" * 80)

# Failsafe: If df is trapped inside main(), recreate it deterministically
if 'df' not in locals() and 'df' not in globals():
    print("\n[!] 'df' not found in global memory. Rebuilding the dataset for the audit...")
    # This uses your existing functions and RANDOM_STATE so it perfectly matches the models
    _temp_df = load_and_prepare_data(DATA_PATH, n_loans=N_LOANS)
    df_audit = engineer_temporal_features(_temp_df)
else:
    df_audit = df.copy()

# ------------------------------------------------
# 1. BASIC DATASET CHECK
# ------------------------------------------------
print("\n[1] DATASET CHECK")
print(f"Total rows in dataset: {len(df_audit):,}")
print(f"Unique loans:          {df_audit['loan_sequence_number'].nunique():,}")

if 'next_month_serious_delinquency' in df_audit.columns:
    print(f"Positive targets:      {int(df_audit['next_month_serious_delinquency'].sum()):,}")
    print(f"Positive rate:         {df_audit['next_month_serious_delinquency'].mean()*100:.4f}%")

# ------------------------------------------------
# 2. FEATURE CHECK
# ------------------------------------------------
print("\n[2] ENGINEERED FEATURE CHECK")
temporal_features = [
    'prev_month_delinquency',
    'delinq_count_6m',
    'upb_pct_change_1m'
]

for feature in temporal_features:
    if feature in df_audit.columns:
        print(f"✓ {feature}")
    else:
        print(f"⚠ MISSING: {feature}")

# ------------------------------------------------
# 3. PREVIOUS-MONTH DELINQUENCY CHECK
# ------------------------------------------------
print("\n[3] TEMPORAL FEATURE LEAKAGE CHECK")
if 'prev_month_delinquency' in df_audit.columns and 'changeDelinquency' in df_audit.columns:

    check_df = df_audit.sort_values(['loan_sequence_number', 'monthly_reporting_period']).copy()

    # Check if 'prev_month' perfectly matches the shift of the current month
    expected_prev = (
        check_df
        .groupby('loan_sequence_number')['changeDelinquency']
        .shift(1)
        .fillna(0)
    )

    actual_prev = check_df['prev_month_delinquency']

    prev_check = np.allclose(
        expected_prev.to_numpy(),
        actual_prev.to_numpy(),
        equal_nan=True
    )

    if prev_check:
        print("✓ prev_month_delinquency correctly strictly uses the previous month (No Leakage)")
    else:
        print("⚠ WARNING: prev_month_delinquency mismatch - Potential Leakage!")
else:
    print("⚠ Could not perform previous-month check because required columns are unavailable.")

# ------------------------------------------------
# 4. TEMPORAL ORDER CHECK
# ------------------------------------------------
print("\n[4] TEMPORAL ORDER CHECK")
if 'month_diff_days' in df_audit.columns:
    bad_gaps = df_audit[df_audit['month_diff_days'] > 32]
    print(f"Irregular transitions (>32 days): {len(bad_gaps):,}")
    if len(bad_gaps) == 0:
        print("✓ No irregular monthly transitions detected")
else:
    print("⚠ MISSING: 'month_diff_days' column not generated")

# ------------------------------------------------
# 5. ACTUAL DATE DISTRIBUTION
# ------------------------------------------------
print("\n[5] DATA DATE RANGE")
if 'date_dt' in df_audit.columns:
    print(f"Earliest month: {df_audit['date_dt'].min().strftime('%Y-%m')}")
    print(f"Latest month:   {df_audit['date_dt'].max().strftime('%Y-%m')}")
    print(f"Unique months:  {df_audit['date_dt'].nunique()}")
else:
    print("⚠ MISSING: 'date_dt' column")

# ------------------------------------------------
# 6. TARGET DISTRIBUTION BY MONTH
# ------------------------------------------------
print("\n[6] TARGET DISTRIBUTION OVER TIME")
if 'date_dt' in df_audit.columns and 'next_month_serious_delinquency' in df_audit.columns:
    monthly_target = (
        df_audit.groupby('date_dt')['next_month_serious_delinquency']
          .agg(['count', 'sum', 'mean'])
          .rename(columns={'count': 'Total_Rows', 'sum': 'Defaults', 'mean': 'Default_Rate'})
    )

    print("\nFirst 5 months:")
    print(monthly_target.head().round(4))

    print("\nLast 5 months:")
    print(monthly_target.tail().round(4))
else:
    print("⚠ MISSING: Required columns for monthly distribution check")

# ------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------
print("\n" + "=" * 80)
print("INTEGRITY AUDIT COMPLETE")
print("=" * 80)

print("""
Checks performed:
✓ Dataset size & Loan count
✓ Positive-event count & rate
✓ Engineered feature existence
✓ Previous-month delinquency construction (Leakage Check)
✓ Monthly temporal continuity
✓ Overall date range
✓ Target distribution over time
""")


FINAL PIPELINE INTEGRITY AUDIT

[!] 'df' not found in global memory. Rebuilding the dataset for the audit...
1. LOADING AND PREPARING DATA
Reading: /content/merged_mortgage_2013.csv
Original shape: (870258, 63)
Dropped 16 columns.
Working sample: 870,258 rows
Unique loans: 9,303

Target distribution:
next_month_serious_delinquency
0    857550
1      3219
Name: count, dtype: int64
next_month_serious_delinquency
0    99.626
1     0.374
Name: percent, dtype: float64

2. TEMPORAL FEATURE ENGINEERING
Irregular monthly transitions (>32 days): 0
Final positive rate: 0.3740%

Data-quality report:
Rows                           860769.00000
Unique_Loans                     9300.00000
Positive_Target_Rate                0.00374
Missing_Target                      0.00000
Irregular_Month_Transitions         0.00000
Missing_Normalized_UPB              0.00000
Missing_UPB_Momentum                0.00000
Missing_Credit_Score                0.00000

[1] DATASET CHECK
Total rows in dataset: 860,769
U